<a href="https://colab.research.google.com/github/harshita1804/ml-systems-journal/blob/transformers/transformer/gpt-dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-11-28 21:48:02--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2025-11-28 21:48:02 (22.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
# read input and inspect
with open('input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

print(len(text))

1115394


In [5]:
chars = sorted(list(set(text))) #list of set og unique chars in the text
vocab_size = len(chars)
print(''.join(chars)) # prints the vocab as string
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [7]:
stoi = {ch:i for i, ch in enumerate(chars)} # encoder mapping
itos = {i:ch for i, ch in enumerate(chars)} # decoder mapping

encode = lambda s: [stoi[c] for c in s] # ENCODER
decode = lambda d: ''.join([itos[i] for i in d]) #DECODER

print(encode("hello world"))
print(decode(encode("hello world")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [8]:
# now we will  use pytorch to encode this data
import torch

encoded_data = torch.tensor(encode(text), dtype = torch.long)
print(encoded_data.shape, encoded_data.dtype)
print(encoded_data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [10]:
# train validation split -- this will help us understand how much our model is overfitting

n = int(0.9*(len(encoded_data)))
train = encoded_data[:n] # first 90% is training data
val = encoded_data[n:]

In [13]:
block_size = 8
train[:block_size+1] # this has multiple examples packed into it.
#In this training block make we simultaneous make it train for 8 examples since we are traning it to make a prediction at each of these 8 positions

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [15]:
x_train = train[:block_size]
y_train = train[1: block_size+1]

for i in range(block_size):
  context = x_train[:i+1]
  ouput = y_train[i]
  print(f"for input {context}, output is {ouput}")
  # we train on all of the 8 examples so that transformer
  #can predict for things as small as 1 char of the contrext window to as big as the context window and be able to predict

for input tensor([18]), output is 47
for input tensor([18, 47]), output is 56
for input tensor([18, 47, 56]), output is 57
for input tensor([18, 47, 56, 57]), output is 58
for input tensor([18, 47, 56, 57, 58]), output is 1
for input tensor([18, 47, 56, 57, 58,  1]), output is 15
for input tensor([18, 47, 56, 57, 58,  1, 15]), output is 47
for input tensor([18, 47, 56, 57, 58,  1, 15, 47]), output is 58


In [18]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
  data = train if split == 'train' else val
  ix = torch.randint(len(data)-block_size, (batch_size,)) #random sampling of input sequences
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+1+block_size] for i in ix])
  return x, y

xb, yb = get_batch('train')
print(xb.shape)
print(xb)
print(yb.shape)
print(yb)

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    output = yb[b, t]
    print(f"when input is{context} output is {output}")


torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input istensor([24]) output is 43
when input istensor([24, 43]) output is 58
when input istensor([24, 43, 58]) output is 5
when input istensor([24, 43, 58,  5]) output is 57
when input istensor([24, 43, 58,  5, 57]) output is 1
when input istensor([24, 43, 58,  5, 57,  1]) output is 46
when input istensor([24, 43, 58,  5, 57,  1, 46]) output is 43
when input istensor([24, 43, 58,  5, 57,  1, 46, 43]) output is 39
when input istensor([44]) output is 53
when input istensor([44, 53]) output is 56
when input istensor([44, 53, 56]) output is 1
when input istensor([44, 53, 56,  1]) output is 58
when input istensor([44

In [27]:
#Bigram language model

import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    # we are creating a 65x65 token embedding table
    # now when we pass the idx every single number in our idx will go to the embedding table and pluck out the corresponding row
    #like.. the element 24 will pluck out the 24th row from the table
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
  #idx -- input xb
  def forward(self, idx, targets = None):
    logits = self.token_embedding_table(idx) #(B,T,C) -- Pytorch then arranges the collection of rows picked out from the table in the form of batch*time*channel
    # in our case batch = 4 , time = block size (think chunk of time) and Channel = vocab size

    if(targets is None) :
      loss = None
    else:
      B, T, C = logits.shape #( Pytorch needs  B, C, T -- it needs channels to be second dim  -- s we will flatten b*t)
      logits  = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss # scores for next char in sequence

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(idx)

      logits = logits[:, -1, :] # B, T, C becomes B, C

      probs = F.softmax(logits, dim = -1) # beconmes B,C, dim = -1 apply sopftmax across rows but from last dim, which is C in our case
      # sample from dist
      idx_next = torch.multinomial(probs, num_samples=1) #(B, 1)

      idx = torch.cat((idx, idx_next), dim = 1) # B, T+1
    return idx

m = BigramLanguageModel(vocab_size)
out, loss = m(xb, yb)
print(out.shape)
print(out, loss)

print(decode(m.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 100)[0].tolist()))

torch.Size([32, 65])
tensor([[-1.5101, -0.0948,  1.0927,  ..., -0.6126, -0.6597,  0.7624],
        [ 0.3323, -0.0872, -0.7470,  ..., -0.6716, -0.9572, -0.9594],
        [ 0.2475, -0.6349, -1.2909,  ...,  1.3064, -0.2256, -1.8305],
        ...,
        [-2.1910, -0.7574,  1.9656,  ..., -0.3580,  0.8585, -0.6161],
        [ 0.5978, -0.0514, -0.0646,  ..., -1.4649, -2.0555,  1.8275],
        [-0.6787,  0.8662, -1.6433,  ...,  2.3671, -0.7775, -0.2586]],
       grad_fn=<ViewBackward0>) tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [28]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [31]:
batch_size = 32

for steps in range(10000):

  xb, yb = get_batch('train')

  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.4386227130889893


In [33]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 1000)[0].tolist()))


OMadgad a
K:
INGLIris b,

WI sets watineak a ntrourow t, ord myvespo my k tonorofr prs, toul panou lelal nnch, y w somesthal LIN:
ATh uak awathiofe spece Whifown s Jus PALe ks,

be tigaso ymay, atolonke oo, pr wh cam:
Hes I ho d ctheatir be sugswhold-bele ouplies,
I or, ceshay;
F:
Thor ishelithe bl, ainoproulll herdetowith, be, wir imakel is; t,

Whis,
T:
NThspor witoosowererines
I d INTh
TESENestandot llagg thin. elo:
t ros twillin le yow:
NI BYon nga mo pro ofend dind?
CHA:

BDYorwe Is CARESed hinisoro owred fin creitod ssiuroris cen fo ctoman thelowim arie, ts
And.
MOLI'swstr.
Anthaithalf test metis:

WA: wn t wofod I
WARG aulernutu butemyofitoprfo whenieligehin m'sond m Me, t ure bont thomano'd

GSiroto d ll t erupord ce R:

VINVR:
SThaie ance theohmfif he wastyo a nderoasts l amauchourstille sthifimy po ant II towarangad, ud RLI d me, and ladhat ahind,
Themevelviesthe'de ngr ud prconos it a frimowhion'edeteist, hacey weron le'theayolllefovild athe ce! oungleasitasend devibathe pe

#Mathematical trick in self attention

In [51]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T, C)
x.shape # 8 tokens in a batch

torch.Size([4, 8, 2])

In [52]:
#averaging till token t so that it talks to past tokens ONLY

xbow = torch.zeros((B,T,C))

for b in range(B):
  for t in range(T):
    x_prev = x[b, :t+1] #t,C
    xbow[b,t]= torch.mean(x_prev,0) #dim=0 means: reduce across the first dimensionm; It takes the mean DOWN the rows (across T), for every column (C). so output shape is C,




In [53]:
wei = torch.tril(torch.ones(T,T))
wei = wei/torch.sum(wei, 1, keepdim= True)
xbow2 = wei @ x #(B,T,T (broadcasting happens here) , B,T,C)


In [58]:
torch.allclose(xbow, xbow2, atol=1e-04)

True

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [44]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3,3))
a = a/torch.sum(a, 1, keepdim=True ) # helps in averaging
b = torch.randint(0,10,(3,2)).float()
c = a @ b

In [45]:
c

tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])

In [42]:
a.shape

torch.Size([3, 3])

In [61]:
#v3
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T,T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1) # softmax over every row -- normalizes
xbow3 = wei @ x

In [62]:
torch.allclose(xbow, xbow3, atol=1e-04)

True